# trying out detection of Tb
- what this nbook does:
    - Nasa tb, IMERG and CCIC loaded, clip to tropics, resampled to IMERG grid
    - tobac application:
        - detect on Tb (Nasa-Tb)
        - segmentate on Tb 
        - bulk stats of Precip (precip)
        - bulk stats of IWP (CCIC)
        - bulk stats on tb
    - => save
        - right now the features are saved at detection, segmentation and bulk stats=> if all executed in one go=> only the last one (bc they all contain the same just with added infos). final file: "features_segmentation_inkl_bulkstats_{date}"
     

- input:
    - Nasa Tb, Imerg, CCIC 
    - date
- adapt:
    - output dir for seg results data
    - tobac parameters for detection and segmentation
- output:
    - Segmentation mask '{savedir}/features_{date}.h5'
    - segmentation features (see note above) 'features_segmentation_inkl_bulkstats_{date}.nc'


In [1]:
import numpy as np
import xarray as xr

# 
from pathlib import Path

# Import tobac itself:
import tobac
import tobac.utils

print("using tobac version", str(tobac.__version__))

# load tools
import sys 
sys.path.append("../tools")
import handle_OBS_data

/work/bb1376/user/leonie/conda/envs/MA_thesis/lib/python3.11/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/work/bb1376/user/leonie/conda/envs/MA_thesis/lib/python3.11/site-packages/tobac/utils/decorators.py:536: UserWarning: Numba not able to be imported; periodic boundary calculations will be slower.Exception raised: ImportError('Numba needs NumPy 2.3 or less. Got NumPy 2.4.')
  warnings.warn(


using tobac version 1.6.2


In [2]:
#Set up directory to save output and plots:
savedir=Path("../output/tobactryoutsdata")
if not savedir.is_dir():
    savedir.mkdir()

# load data

In [3]:
# set date
date = "2006-06-06-1800"

# load data (load, allign coords, selects Tb and precip variable)
tb = handle_OBS_data.load_clean_crop_tb(date,32)

# load data
precip = handle_OBS_data.load_clean_crop_imerg(date,32)


# load data
iwp = handle_OBS_data.load_clean_crop_ccic(date,32)


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

Tb data loaded, cropped, 'Tb' selected, assigned attrs, drop time dim


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

/work/bb1376/user/leonie/master_thesis/nbooks/../tools/get_data.py:138: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'julian', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.
  imerg_clean["time"] = imerg_clean.indexes["time"].to_datetimeindex(time_unit = "ns")


IMERG data loaded, cleaned, cropped, 'precipitation' selected, assigned attrs, drop time dim
CCIC data loaded, cleaned, cropped, 'tiwp' selected, assigned attrs, drop time dim


# Prepare Data
In the next steps, we
- (regular grid)
- resmaple to imerg data so everything has same resolution (for bulk statiustics needed) 
- convert the dataset into an Iris Cube (which is taken as tobac input)


In [4]:
# resample (data, precip)
tb_res = handle_OBS_data.resample_grids(tb,precip)  
iwp_res = handle_OBS_data.resample_grids(iwp,precip)    

# to iris cubes
tb_iris = tb_res.to_iris()
precip_iris = precip.to_iris()
iwp_iris = iwp_res.to_iris()



# expand time dim for tobac input (collapsed in the loading functions)
# import iris
tb_iris = iris.util.new_axis(tb_iris, scalar_coord='time')
precip_iris = iris.util.new_axis(precip_iris, scalar_coord='time')
iwp_iris = iris.util.new_axis(iwp_iris, scalar_coord='time')


NameError: name 'iris' is not defined

In [ ]:
tb_res


# tobac detection on Tb image
- use multiple threshold on BT data

### detection parameter

In [ ]:
# set horizontal grid spacing [m] and temporal resolution [s]
dxy = 4000
#dt = 1800 # 0.5h # for linking

# detection para (Tb)
feature_detection_params = dict()
feature_detection_params["threshold"] = [225,245,250]
feature_detection_params["target"] = "minimum"
# minimum number of pixels below threshold
feature_detection_params["n_min_threshold"] = 10
# Gaussian smoothing before detection
feature_detection_params["sigma_threshold"] = 1
#feature_detection_params["min_distance"]=

### feature detection

In [ ]:
# Perform feature detection:
print('starting feature detection')
features = tobac.feature_detection.feature_detection_multithreshold(
    tb_iris, dxy, **feature_detection_params
)

features.to_hdf(savedir / f'features_{date}.h5', key= 'table')
print('feature detection performed and saved')
features

### explore & filter results

In [ ]:
# filter for thres>20
filtered_features = features[features["threshold_value"] == 225]
filtered_features

# Segmentation of Anvils (on BT)
- only on filitered features

### set segmentation parameters
- 235K Bolot 2026: "The outer boundaries of the tracked DCSs are set by a 235 K brightness temperature (BT) threshold, which encompasses both the convective core of the DCSs – the inner part of the DCS doing the vertical transport – and the thick cirrus cloud shield immediately attached to it"
  
- Kukulies 2025: "We define MCSs following the criteria established by Feng et al. (2021) and applied  209 in two comprehensive MCS intercomparison studies (A. F. Prein et al., 2024; Feng et al.,  210 2024): a cloud shield with Tb ≤ 240 K spanning at least 40,000 km2 and a precipitation  211 peak ≥ 10 mm h−1, persisting for at least 4 continuous hours. Additionally, the detected  212 clouds must include a Tb value < 225 K, indicating a cold core, and a precipitation vol213 ume ≥ 20, 000 km2 mm h−1 at least once during the system’s lifetime."
 
- Feng 2023: Briefly, a DCS is defined as contiguous areas with Tb < 241 K (cold cloud shield, or CCS) that may contain convective cores (Tb < 225 K) and surface precipitation. These Tb were used in Feng, Leung, et al. (2021) to define DCSs. A detect and dilate approach is used to expand the convective cores to encompass the surrounding cold anvil clouds to identify and track DCS. From the tracked DCSs, MCSs are defined as.  1. CCS area >4 × 104 km2 containing a precipitation feature (PF, contiguous area with rain rate >2 mm hr−1) with major axis length >100 km, 2. Various PF characteristics exceed thresholds in Table S2 in Supporting Information S1, and 3. Both criteria 1 and 2 are met continuously for longer than 4 hr."

- feng 2021: "As illustrated in Figure 1a, cold cloud cores (Tb < 225 K) are first identified and labeled. They are then spread out iteratively to reach a secondary Tb threshold <241 K to define a CCS"

- esmaili 2016: "To capture a variety of cloud clusters, we used a single 235 K brightness temperature threshold, which in the upper atmosphere corresponds to a height of roughly 10 km (9 km) in the tropics (midlatitudes), which is well into the free atmosphere. Additionally, we applied a minimum size threshold of 100 contiguous pixels (1600 km2 at the equator) at all time steps, thus limiting the study to events at the upper bounds of mesoscale or larger. We excluded smaller-scale events because they would be more suitable for regional studies"

- senf 2018: "A BT10.8 threshold of 230 K has been chosen, which roughly corresponds to a height of 11 km in a standard tropical atmosphere (Machado et al. 1992). This temperature threshold is identical to the threshold considered in Schroeder et al. (2009) and Bennartz and Schroeder (2012) to assess the distribution of convective activity over Africa and the tropical Atlantic. With BTs lower than 230 K, only the deep convective cores and the upper parts of thick convective anvil clouds are taken into account. Quite a large diversity on threshold choices can be found in the literature, however [e.g., see discussion in Bennartz and Schroeder (2012"

In [ ]:
parameters_segmentation = dict()
parameters_segmentation["method"] = "watershed"
parameters_segmentation["threshold"] = 235
parameters_segmentation["target"] = "minimum"

### perform segmentation

In [ ]:
features = filtered_features

# Perform segmentation and save results:
print('Starting segmentation.')
segmentation_mask, segmentation_features = tobac.segmentation.segmentation(
    features, tb_iris, dxy=2000, **parameters_segmentation
)

# segmentation_mask, segmentation_features = tobac.segmentation_2D(
#     features, tb_iris, dxy=2000, **parameters_segmentation, statistic = statistics
# )



### save segmentation results

In [ ]:
print('segmentation performed, start saving results to files')
segmentation_mask_ds = xr.DataArray.from_iris(segmentation_mask)
segmentation_mask_ds.to_netcdf(savedir / f'mask_segmentation_{date}.nc', encoding={"segmentation_mask":{"zlib":True, "complevel":4}})

#tmp = xr.DataArray.from_iris(segmentation_features)
segmentation_features.to_hdf(savedir / f'features_segmentation_{date}.h5', key = 'table')
print('segmentation performed and saved')

# bulk statistics as post processing
- use "iwp_iris" loaded before

## Tb

In [ ]:
from tobac.utils import get_statistics_from_mask

statistics = {}
statistics['min_tb'] = np.min

features_with_tb_stats = get_statistics_from_mask(features, segmentation_mask, tb_iris, statistic=statistics)

## iwp

In [ ]:
from tobac.utils import get_statistics_from_mask

statistics = {}
statistics['mean_iwp'] = np.mean
statistics['total_iwp'] = np.sum
statistics['max_iwp'] = np.max


features_with_iwp_stats = get_statistics_from_mask(features, segmentation_mask, iwp_iris, statistic=statistics)

## Precip


In [ ]:
from tobac.utils import get_statistics_from_mask

statistics = {}
statistics['mean_precip'] = np.mean
statistics['total_precip'] = np.sum
statistics['max_precip'] = np.max


features_with_precip_stats = get_statistics_from_mask(features, segmentation_mask, precip_iris, statistic=statistics)

## add area 

In [ ]:
area = tobac.calculate_area(features, segmentation_mask)
area

## merge into one

In [ ]:
df_merged = segmentation_features.merge(
    features_with_iwp_stats[['feature', 'mean_iwp', 'total_iwp', 'max_iwp']],
    on='feature',
    how='left'
)
df_merged = df_merged.merge(
    features_with_tb_stats[['feature', 'min_tb']],
    on='feature',
    how='left'
)

df_merged = df_merged.merge(
    features_with_precip_stats[['feature', 'mean_precip', 'total_precip', 'max_precip']],
    on='feature',
    how='left'
)

df_merged = df_merged.merge(
    area[['feature', 'area']],
    on='feature',
    how='left'
)

df_merged.to_hdf(savedir / f'features_segmentation_inkl_bulkstats_{date}.h5', key = 'table')
print('features with bulk statistics saved')

In [ ]:
df_merged

In [ ]:
filtered_features